## Data Exploration

In [2]:
import pandas as pd
import numpy as np

# Load data

df = pd.read_excel("../data/raw/ethiopia_fi_unified_data.xlsx")
ref = pd.read_excel("../data/raw/reference_codes.xlsx")

print(df['record_type'].value_counts())
print(df.columns.tolist())
df.head()

record_type
observation    30
event          10
target          3
Name: count, dtype: int64
['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


### Step 1.2: Understand Structure

In [3]:
# Separate by record type
obs = df[df['record_type'] == 'observation']
events = df[df['record_type'] == 'event']
impact_links = df[df['record_type'] == 'impact_link']
targets = df[df['record_type'] == 'target']

print("Observations:", obs.shape)
print("Events:", events.shape)
print("Impact Links:", impact_links.shape)
print("Targets:", targets.shape)

Observations: (30, 34)
Events: (10, 34)
Impact Links: (0, 34)
Targets: (3, 34)


Check temporal range:

In [4]:
obs['observation_date'] = pd.to_datetime(obs['observation_date'])
print(obs['observation_date'].min(), obs['observation_date'].max())

2014-12-31 00:00:00 2025-12-31 00:00:00


List indicators:

In [5]:
print(obs['indicator_code'].unique())

<StringArray>
[     'ACC_OWNERSHIP',     'ACC_MM_ACCOUNT',         'ACC_4G_COV',
     'ACC_MOBILE_PEN',          'ACC_FAYDA',      'USG_P2P_COUNT',
      'USG_P2P_VALUE',      'USG_ATM_COUNT',      'USG_ATM_VALUE',
      'USG_CROSSOVER', 'USG_TELEBIRR_USERS', 'USG_TELEBIRR_VALUE',
    'USG_MPESA_USERS',   'USG_MPESA_ACTIVE',    'USG_ACTIVE_RATE',
    'AFF_DATA_INCOME',        'GEN_GAP_ACC',       'GEN_MM_SHARE',
     'GEN_GAP_MOBILE']
Length: 19, dtype: str


### Step 1.3: Enrich Dataset

In [6]:
# New observations
new_obs = pd.DataFrame([
    {
        "record_type": "observation",
        "pillar": "enabler",
        "indicator_code": "INFRA_SMARTPHONE_PEN",
        "value_numeric": 45.0,
        "observation_date": "2024-01-01",
        "source_name": "GSMA Mobile Economy Sub-Saharan Africa 2024",
        "source_url": "https://www.gsma.com/mobilefordevelopment/wp-content/uploads/2024/06/GSMA_MobileEconomy2024_SSA_Eng.pdf",
        "confidence": "high",
        "notes": "Estimated smartphone penetration in Ethiopia"
    },
    {
        "record_type": "observation",
        "pillar": "enabler",
        "indicator_code": "INFRA_4G_COVERAGE",
        "value_numeric": 80.0,
        "observation_date": "2024-01-01",
        "source_name": "Ethio Telecom Annual Report 2024",
        "source_url": "https://www.ethiotelecom.et",
        "confidence": "medium",
        "notes": "4G population coverage"
    }
])

# New events
new_events = pd.DataFrame([
    {
        "record_type": "event",
        "category": "digital_id",
        "event_name": "Fayda Digital ID Launch",
        "event_date": "2023-09-01",
        "source_name": "Fayda Digital ID Portal",
        "source_url": "https://id.gov.et",
        "confidence": "high",
        "notes": "National digital ID system launched"
    },
    {
        "record_type": "event",
        "category": "infrastructure",
        "event_name": "EthSwitch Interoperability Go-Live",
        "event_date": "2022-05-01",
        "source_name": "EthSwitch Press Release",
        "source_url": "https://ethswitch.com",
        "confidence": "high",
        "notes": "Enabled P2P transfers across mobile money providers"
    }
])

# Assign temporary IDs (ensure no conflict with existing)
max_id = df['id'].max() if 'id' in df.columns else 0
if 'id' not in new_events.columns:
    new_events['id'] = range(max_id + 1, max_id + len(new_events) + 1)
    max_id = new_events['id'].max()

# New impact links (link to event IDs)
new_impact = pd.DataFrame([
    {
        "record_type": "impact_link",
        "parent_id": new_events.iloc[0]['id'],  # Fayda
        "pillar": "access",
        "related_indicator": "ACC_OWNERSHIP",
        "impact_direction": "+",
        "impact_magnitude": 2.0,  # +2 percentage points
        "lag_months": 12,
        "evidence_basis": "Based on India's Aadhaar impact on bank account ownership (World Bank, 2018)",
        "confidence": "medium"
    },
    {
        "record_type": "impact_link",
        "parent_id": new_events.iloc[1]['id'],  # EthSwitch
        "pillar": "usage",
        "related_indicator": "USG_DIGITAL_PAYMENT",
        "impact_direction": "+",
        "impact_magnitude": 5.0,
        "lag_months": 6,
        "evidence_basis": "Interoperability typically boosts transaction volume and usage (GSMA, 2022)",
        "confidence": "medium"
    }
])

Step 1.4: Combine and Save

In [8]:
# Ensure all have same columns
all_cols = df.columns.tolist()
for new_df in [new_obs, new_events, new_impact]:
    for col in all_cols:
        if col not in new_df.columns:
            new_df[col] = np.nan

# Concatenate
enriched_df = pd.concat([df, new_obs, new_events, new_impact], ignore_index=True)

# Save
enriched_df.to_csv("../data/processed/ethiopia_fi_unified_data_enriched.csv", index=False)